
# 🔐 Nemo Safe Synthesizer Tutorial: The Basics

#### What you'll learn

In this notebook, we'll explore the fundamentals of the NeMo Safe Synthesizer: PII replacement, training on a sample dataset, generating synthetic data, and evaluating quality and privacy.

This library supports numeric, categorical, and text fields within the training data and generates realistic synthetic data that mirrors the structure of your data. A full run takes ~15 minutes on an A100; an H100 is faster.

### 🖥️ Prerequisites

This notebook is intended to run on a **GPU**. We recommend an **H100**; minimum **A100**.




### ⚡ Colab Setup

Run the cell below to install Nemo Safe Synthesizer (engine and CUDA 12.8) and the `datasets` library for the sample dataset.

In [ ]:
%%capture
# SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

!uv pip install "nemo-safe-synthesizer[engine,cu128]" --index https://flashinfer.ai/whl/cu128 --index https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match
!uv pip install datasets



### 🔑 Set the inference API key for column classification

NeMo Safe Synthesizer uses an LLM‑based column classifier to automatically infer column types and improve PII detection accuracy. To enable this feature, set `NSS_INFERENCE_KEY` (the inference endpoint defaults to the NVIDIA integrate URL. You can obtain an API key from [build.nvidia.com](https://build.nvidia.com/settings/api-keys)). Setting this value is optional but strongly recommended.


In [ ]:
import os
import getpass

# Setting NSS_INFERENCE_KEY is optional but strongly recommended for PII replacement.
if "NSS_INFERENCE_KEY" not in os.environ:
    os.environ["NSS_INFERENCE_KEY"] = getpass.getpass("Paste inference API key (or press Enter to skip): ")
if os.environ.get("NSS_INFERENCE_KEY"):
    print("NSS_INFERENCE_KEY is set")
else:
    print(
        "NSS_INFERENCE_KEY is not set. "
        "We strongly recommend setting a key."
    )

### 📥 Load and preview sample dataset

Load a tabular dataset—in this example, the [clinc_oos](https://huggingface.co/datasets/clinc/clinc_oos) from Huggingface—and preview the first few rows. NeMo Safe Synthesizer will use this DataFrame as its training data.

This dataset includes a text column and a categorical intent label, making it a good demonstration of multi-type synthesis.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("clinc/clinc_oos", "small")
df = dataset["train"].to_pandas()  # type: ignore[union-attr]
df.head()  # type: ignore[union-attr]




###  ⚙️ Create and run Safe Synthesizer job

Create the Safe Synthesizer builder and attach your DataFrame. 
Run the pipeline with `run()`, which performs data processing, PII replacement, training, generation, and evaluation in a single call. Results are available on `builder.results`.

 Please refer to the [configuration docs](https://github.com/NVIDIA-NeMo/Safe-Synthesizer/blob/main/docs/user-guide/configuration.md) for the full list of options.



In [ ]:
from nemo_safe_synthesizer.sdk.library_builder import SafeSynthesizer


# To disable PII replacement for the run, chain `.with_replace_pii(enable=False)` on the builder before `run()`.
builder = SafeSynthesizer().with_data_source(df)

builder.run()
results = builder.results

### 📤 Retrieve synthetic data

Inspect the generated synthetic data including row count and preview of the first rows.

In [ ]:
synth = results.synthetic_data
print(f"Number of synthetic rows: {len(synth)}")
synth.head()

### 🛡️ Review evaluation report

The pipeline computes both quality and privacy metrics. The summary includes timing information and overall scores, while the full evaluation report is rendered as an HTML document.

In [ ]:
import json

print("Summary (timing and scores):")
print(json.dumps(results.summary.model_dump(), indent=2))

In [ ]:
# Download the full HTML evaluation report
if results.evaluation_report_html:
    report_path = "evaluation_report.html"
    with open(report_path, "w") as f:
        f.write(results.evaluation_report_html)
    print(f"The HTML evaluation report is saved in {report_path}.")